# Worksheet 3 · Querying and Reasoning with EQL

**Advanced Methods of AI · WiSe 2026/2027 · Tasks 2 to 4**

Last week you *represented* the pizza world: classes for toppings and pizzas, a few objects, and four
hand-written functions (`is_cheesy`, `is_vegetarian`, ...). This week you *ask* it things.

The setting: it is Thursday evening in a small pizzeria. Seven pizzas are on the counter, three tables have
ordered, and one guest has a milk allergy. You are building the kitchen assistant.

**How to work.** Run the cells from top to bottom. Cells marked `TODO` are yours. Every cell runs as it is,
so you can always continue; an unfinished query prints `TODO`. For the **Predict** cells, write down what you
expect *before* you run them. That is where the learning happens.

A query in EQL (the *Entity Query Language* of KRROOD) is an ordinary Python expression. Nothing is computed
when you write it. It is evaluated when you ask for its results, here with `show(...)`.

## 0 · The knowledge base

The toppings and pizzas of `pizza.owl` as dataclasses, as in Worksheet 2. New this week are `Guest` and
`Order`, because a kitchen needs to know who gets what. **Run the next three cells; do not edit them.**

In [ ]:
from __future__ import annotations

import sqlite3
from dataclasses import dataclass, field
from enum import Enum

from krrood.entity_query_language.factories import (
    a, an, the, variable, flat_variable, entity, set_of,
    or_, not_, exists, for_all, contains,
    inference, deduced_variable, add, refinement, alternative,
    HasType, length, count,
)
from krrood.entity_query_language.exceptions import MultipleSolutionFound
from krrood.entity_query_language.predicate import Predicate, symbolic_function
from krrood.entity_query_language.verbalization.vocabulary.english import Prepositions
from krrood.entity_query_language.verbalization.vocabulary.parts_of_speech import (
    Adjective, Copula, Noun, clause,
)

In [ ]:
class Spiciness(Enum):
    MILD = 1
    MEDIUM = 2
    HOT = 3


class Allergen(Enum):
    # Three of the 14 allergens a restaurant in the EU has to declare.
    GLUTEN = "gluten"
    MILK = "milk"
    FISH = "fish"


# ---- toppings (the class hierarchy of pizza.owl) ----

@dataclass(eq=False, repr=False)
class PizzaTopping:
    spiciness: Spiciness = Spiciness.MILD

    def __repr__(self):
        return type(self).__name__


class CheeseTopping(PizzaTopping): ...
class MozzarellaTopping(CheeseTopping): ...
class ParmesanTopping(CheeseTopping): ...

class VegetableTopping(PizzaTopping): ...
class TomatoTopping(VegetableTopping): ...
class OliveTopping(VegetableTopping): ...
class CaperTopping(VegetableTopping): ...
class MushroomTopping(VegetableTopping): ...

@dataclass(eq=False, repr=False)
class JalapenoPepperTopping(VegetableTopping):
    spiciness: Spiciness = Spiciness.HOT

@dataclass(eq=False, repr=False)
class HotGreenPepperTopping(VegetableTopping):
    spiciness: Spiciness = Spiciness.HOT

class MeatTopping(PizzaTopping): ...
class HamTopping(MeatTopping): ...

@dataclass(eq=False, repr=False)
class PeperoniSausageTopping(MeatTopping):
    spiciness: Spiciness = Spiciness.MEDIUM

class FishTopping(PizzaTopping): ...
class AnchoviesTopping(FishTopping): ...


# ---- pizzas ----

@dataclass(eq=False, repr=False)
class Pizza:
    name: str
    toppings: list[PizzaTopping] = field(default_factory=list)

    def __repr__(self):
        return self.name


class Margherita(Pizza): ...
class Napoletana(Pizza): ...
class AmericanHot(Pizza): ...
class Mushroom(Pizza): ...


# ---- new this week: guests and orders ----

@dataclass(eq=False, repr=False)
class Guest:
    name: str
    avoids: set[Allergen] = field(default_factory=set)

    def __repr__(self):
        return self.name


@dataclass(eq=False, repr=False)
class Order:
    table: int
    guest: Guest
    pizzas: list[Pizza]

    def __repr__(self):
        return f"Order(table {self.table}, {self.guest}, {self.pizzas})"

In [ ]:
# ---- the objects: what is true tonight ----

margherita_1 = Margherita("margherita_1", [MozzarellaTopping(), TomatoTopping()])
napoletana_1 = Napoletana("napoletana_1", [AnchoviesTopping(), CaperTopping(), MozzarellaTopping(),
                                           OliveTopping(), TomatoTopping()])
american_hot_1 = AmericanHot("american_hot_1", [HotGreenPepperTopping(), JalapenoPepperTopping(),
                                                MozzarellaTopping(), PeperoniSausageTopping(),
                                                TomatoTopping()])
mushroom_1 = Mushroom("mushroom_1", [MozzarellaTopping(), MushroomTopping(), TomatoTopping()])
veggie_1 = Pizza("veggie_1", [TomatoTopping(), OliveTopping(), MushroomTopping()])
plain_1 = Pizza("plain_1", [])                          # p3 of Worksheet 2: no toppings at all
unclosed_1 = Pizza("unclosed_1", [MozzarellaTopping()])  # Worksheet 2, Task 3(c)

pizzas = [margherita_1, napoletana_1, american_hot_1, mushroom_1, veggie_1, plain_1, unclosed_1]

ada = Guest("Ada")
ben = Guest("Ben", avoids={Allergen.MILK})
chidi = Guest("Chidi", avoids={Allergen.FISH})
guests = [ada, ben, chidi]

orders = [
    Order(1, ada, [margherita_1, american_hot_1]),
    Order(2, ben, [veggie_1, napoletana_1]),
    Order(3, chidi, [mushroom_1]),
]


# ---- last week's functions (solution of Worksheet 2, Task 3) ----

def is_cheesy(pizza: Pizza) -> bool:
    return any(isinstance(t, CheeseTopping) for t in pizza.toppings)

def is_meaty(pizza: Pizza) -> bool:
    return any(isinstance(t, MeatTopping) for t in pizza.toppings)

def is_vegetarian(pizza: Pizza) -> bool:
    return not any(isinstance(t, (MeatTopping, FishTopping)) for t in pizza.toppings)

def is_interesting(pizza: Pizza) -> bool:
    return len(pizza.toppings) >= 3


def show(title, query):
    # Evaluate a query and print one result per line.
    if query is ...:
        print(f"{title}: TODO")
        return
    results = query.tolist()
    print(f"{title}: {len(results)} result(s)")
    for result in results:
        print("   ", tuple(result.values()) if hasattr(result, "values") else result)

---
## Task 2 · Asking questions

### 2(a) One question, two spellings

*"Which pizzas on the counter have a cheese topping?"* This is the concept
`C1 = ∃hasTopping.CheeseTopping` of Worksheet 2, now asked of real objects.

**Spelling 1, a description.** `a(Pizza)(...)` describes one object by its attributes, and `.from_(pizzas)`
says where to look. The attribute `toppings` is a list; a nested description such as `a(CheeseTopping)()`
means *some element* of that list fits.

In [ ]:
cheesy = a(Pizza)(toppings=a(CheeseTopping)()).from_(pizzas)
show("cheesy pizzas", cheesy)

**Spelling 2, variables and conditions.** This is the first-order formula
`Pizza(p) ∧ ∃t (hasTopping(p, t) ∧ CheeseTopping(t))` almost symbol for symbol. `p` is the free variable,
so the answers are values for `p`.

In [ ]:
p = variable(Pizza, domain=pizzas)        # p ranges over the pizzas on the counter
t = flat_variable(p.toppings)             # t ranges over the toppings of p

cheesy_again = entity(p).where(exists(t, HasType(t, CheeseTopping)))
show("cheesy pizzas (spelling 2)", cheesy_again)

**Your turn.** Descriptions nest as deep as the objects do.

1. `hot_pizzas`: the pizzas with some topping whose spiciness is `Spiciness.HOT`
   (the `SpicyPizza` of Worksheet 2).
2. `hot_orders`: the orders that contain such a pizza. Look at the class `Order`: this goes through *two*
   lists.

In [ ]:
hot_pizzas = a(Pizza)(toppings=a(PizzaTopping)(spiciness=Spiciness.HOT)).from_(pizzas)
hot_orders = an(Order)(
    pizzas=a(Pizza)(toppings=a(PizzaTopping)(spiciness=Spiciness.HOT))
).from_(orders)

show("hot pizzas", hot_pizzas)
show("orders with a hot pizza", hot_orders)

**Solution note.** `a(PizzaTopping)(spiciness=Spiciness.HOT)` reads "some topping that is hot". In DL this is
`Pizza ⊓ ∃hasTopping.(∃hasSpiciness.Hot)`, the nested concept of Worksheet 2, Task 2(b)ii. The order query
nests once more: *some* pizza of the order has *some* hot topping. `american_hot_1` has two hot toppings and
still appears once: a description is either satisfied or not.

### 2(b) *a* or *the*?

A robot that is told *"bring me the pizza with fish"* has to find exactly one object.
`the(...)` is a query that insists on this.

**Predict:** what do the next two cells do?

In [ ]:
show("the fish pizza", the(Pizza)(toppings=a(FishTopping)()).from_(pizzas))

In [ ]:
try:
    show("the cheese pizza", the(Pizza)(toppings=a(CheeseTopping)()).from_(pizzas))
except MultipleSolutionFound as error:
    print("the cheese pizza:", type(error).__name__, "-", error)

**Solution note.** The first cell prints `napoletana_1`. In the second, the query raises
`MultipleSolutionFound` (the cell catches it and prints it): five pizzas fit. That is the right behaviour, not a nuisance. *"The pizza with cheese"* is a definite description
(Russell: there is exactly one such thing), and here it does not refer. A robot should not pick one at
random; it should ask *"which one?"*, which is itself a query: the difference between the candidates.
With no fish pizza on the counter, `the` raises `NoSolutionFound`.

### 2(c) Some, only, none

**Predict** which pizzas each of the next three queries returns, then run the cell. Watch `plain_1`
(no toppings) and `unclosed_1` (only mozzarella).

In [ ]:
p = variable(Pizza, domain=pizzas)
t = flat_variable(p.toppings)

only_vegetables = entity(p).where(for_all(t, HasType(t, VegetableTopping)))
show("Q1  only vegetable toppings", only_vegetables)

exactly_margherita = a(Pizza)(toppings=[a(MozzarellaTopping)(), a(TomatoTopping)()]).from_(pizzas)
show("Q2  a list of two descriptions", exactly_margherita)

no_cheese = entity(p).where(not_(exists(t, HasType(t, CheeseTopping))))
show("Q3  no cheese topping", no_cheese)

**Solution note.**

- **Q1** returns `veggie_1` and `plain_1`. `for_all` over an empty list is true (vacuous truth), exactly as
  p3 was in `C2 = ∀hasTopping.VegetableTopping` last week.
- **Q2** returns only `margherita_1`. A *list* of descriptions describes the *whole* list: first element a
  mozzarella, second a tomato, and nothing else. This is "some + only" in one go, the closure axiom of
  Worksheet 2, Task 2(b)iii. `mushroom_1` has both toppings but also a third, `unclosed_1` has only one.
  Compare with 2(a): a *single* description means "some element".
- **Q3** returns `veggie_1` and `plain_1`. Nobody stated that these pizzas have no cheese. The query
  concludes it because none is in the list: negation as failure, the closed-world assumption.

**Your turn.** Write `vegetarian`: the pizzas with no meat topping and no fish topping, using `exists`.

Hint: use *one* `exists` whose condition is an `or_(...)`.

In [ ]:
vegetarian = entity(p).where(
    not_(exists(t, or_(HasType(t, MeatTopping), HasType(t, FishTopping))))
)

show("vegetarian pizzas", vegetarian)

**Solution note.** Result: `margherita_1`, `mushroom_1`, `veggie_1`, `plain_1`, `unclosed_1`. By De Morgan,
`for_all(t, not_(or_(...)))` is the same query.

The hint matters. A variable is local to an `exists` only if it occurs nowhere else in the query. With two
separate `not_(exists(t, ...))` conditions that share `t`, the variable `t` becomes visible outside and the
query turns into a join over the toppings. If you want two quantifiers, give each its own
`flat_variable(p.toppings)`.

### 2(d) Answers are bindings

A description returns objects. With several variables, an answer is a *binding*: one value for each variable
that makes the conditions true together. `set_of(...)` selects more than one variable. This is a join.

**Predict:** how many rows does the next query return, and why not three?

In [ ]:
o = variable(Order, domain=orders)
ordered = flat_variable(o.pizzas)                 # one pizza of the order
topping = flat_variable(ordered.toppings)         # one topping of that pizza

hot_rows = set_of(o.guest, ordered, topping).where(topping.spiciness == Spiciness.HOT)
show("who gets which hot topping", hot_rows)

**Solution note.** Two rows, both for Ada and `american_hot_1`: one for the jalapeño, one for the green
pepper. A join returns every binding that satisfies the conditions. `hot_orders` in 2(a) returned the order
once because `exists` only asks *whether* there is a hot topping. In SQL this is the difference between
`JOIN` and `WHERE EXISTS (...)`; forgetting it is the classic source of duplicated rows.

**Your turn.** The kitchen wants to know how much work each table is: write `toppings_per_table`, which
returns for each order its table number and the number of toppings on all its pizzas.

Hint: `set_of(o.table, count(topping)).grouped_by(o.table)`. This is `GROUP BY`.

In [ ]:
toppings_per_table = set_of(o.table, count(topping)).grouped_by(o.table)

show("toppings per table", toppings_per_table)

**Solution note.** Table 1 has 7 toppings, table 2 has 8, table 3 has 3. The variables `o`, `ordered` and
`topping` from the previous cell are reused: a variable is a value like any other and can appear in several
queries.

---
## Task 3 · Reasoning: functions, predicates and rules

Ben has a milk allergy. Nothing in the knowledge base says *"napoletana_1 contains milk"*. It has to be
*derived*: mozzarella is a cheese, cheese contains milk, the pizza has that topping. This task derives it in
three ways.

### 3(a) Your functions become vocabulary

Last week's `is_vegetarian` is a plain Python function: you can call it on one pizza. Wrapped in
`symbolic_function`, it can also be called on a *variable*, and then it becomes a condition of a query.

In [ ]:
is_cheesy = symbolic_function(is_cheesy)
is_meaty = symbolic_function(is_meaty)
is_vegetarian = symbolic_function(is_vegetarian)
is_interesting = symbolic_function(is_interesting)

print("on an object:  ", is_vegetarian(margherita_1))
print("on a variable: ", type(is_vegetarian(p)).__name__, "- not a truth value, a piece of a query")

show("vegetarian and interesting", entity(p).where(is_vegetarian(p), is_interesting(p)))

A symbolic function may return any value, not only a truth value. `allergens` computes what a pizza
contains. Every pizza contains gluten because of its base.

**Your turn.** Complete `allergens` so that a pizza with a cheese topping contains `Allergen.MILK` and a
pizza with a fish topping contains `Allergen.FISH`.

In [ ]:
@symbolic_function
def allergens(pizza: Pizza) -> set[Allergen]:
    found = {Allergen.GLUTEN}
    if any(isinstance(topping, CheeseTopping) for topping in pizza.toppings):
        found.add(Allergen.MILK)
    if any(isinstance(topping, FishTopping) for topping in pizza.toppings):
        found.add(Allergen.FISH)
    return found


for pizza in pizzas:
    print(f"{pizza.name:16}", sorted(allergen.value for allergen in allergens(pizza)))

### 3(b) A predicate relates objects

`hasTopping(x, y)` is *stored*: it is the list `toppings`. Whether a pizza is safe for a guest is stored
nowhere; it is *computed* from both. A `Predicate` is a relation defined by a computation. Its fields are
its arguments, `__call__` decides whether it holds, and `_verbalization_fragment_` tells EQL how to say it
in English.

**Your turn.** Complete `__call__`: a pizza is safe for a guest if it contains none of the allergens the
guest avoids.

In [ ]:
@dataclass(eq=False)
class IsSafeFor(Predicate):
    pizza: Pizza
    guest: Guest

    def __call__(self) -> bool:
        return not (allergens(self.pizza) & self.guest.avoids)

    @classmethod
    def _verbalization_fragment_(cls, fields):
        return clause(Noun(fields["pizza"]), Copula(), Adjective("safe"),
                      Prepositions.FOR, Noun(fields["guest"]))


print("napoletana_1 safe for Ben?", bool(IsSafeFor(napoletana_1, ben)))
print("veggie_1 safe for Ben?    ", bool(IsSafeFor(veggie_1, ben)))

Now the predicate is part of the language. Two queries the kitchen needs:

1. `safe_choices` (given): which guest can eat which pizza?
2. `dangerous` (**your turn**): the pairs of an order and one of its pizzas that is *not* safe for the guest
   of that order. Reuse the variables `o` and `ordered` of 2(d).

The second one is a different kind of query. Nobody wants to read its answers; the kitchen wants it to be
**empty**. A query whose answer has to be empty is a *constraint*.

In [ ]:
g = variable(Guest, domain=guests)
safe_choices = set_of(g, p).where(IsSafeFor(p, g))
show("safe choices", safe_choices)

dangerous = set_of(o, ordered).where(not_(IsSafeFor(ordered, o.guest)))
show("DANGEROUS", dangerous)

**Solution note.** `dangerous` has one row: table 2, Ben, `napoletana_1`. Ben avoids milk and the Napoletana
has mozzarella. The constraint is violated, and the query does not only say *that* it is violated: the
binding says *where*. Note `o.guest`: the path from the order to its guest replaces a join.

`safe_choices` has 15 rows: all 7 pizzas for Ada, 2 for Ben (`veggie_1`, `plain_1`) and 6 for Chidi (all
but `napoletana_1`).

### 3(c) Rules produce new facts

A predicate computes its answer each time it is asked. A **rule** writes the conclusion down as a new object
that later queries can use. `inference(AllergenWarning)(...)` creates one `AllergenWarning` for every
binding that satisfies the description.

The first rule is given: *if a pizza has some cheese topping, it carries a milk warning.*

**Your turn.** Write the rule for fish, and the rule that *every* pizza carries a gluten warning (a rule
without a condition: use the variable `p`).

In [ ]:
@dataclass(eq=False, repr=False)
class AllergenWarning:
    pizza: Pizza
    allergen: Allergen

    def __repr__(self):
        return f"Warning({self.pizza}: {self.allergen.value})"


cheesy = a(Pizza)(toppings=a(CheeseTopping)()).from_(pizzas)
milk_warnings = inference(AllergenWarning)(pizza=cheesy, allergen=Allergen.MILK).tolist()

fishy = a(Pizza)(toppings=a(FishTopping)()).from_(pizzas)
fish_warnings = inference(AllergenWarning)(pizza=fishy, allergen=Allergen.FISH).tolist()
gluten_warnings = inference(AllergenWarning)(pizza=p, allergen=Allergen.GLUTEN).tolist()

warnings = milk_warnings + fish_warnings + gluten_warnings
print(len(warnings), "warnings:", warnings)

The warnings are objects, so they are part of the knowledge base now and can be queried like pizzas. The
constraint of 3(b) again, this time as a join over the *derived* facts:

In [ ]:
w = variable(AllergenWarning, domain=warnings)

dangerous_by_rule = set_of(o, w).where(
    w.pizza == ordered,                        # the warning is about a pizza of the order ...
    contains(o.guest.avoids, w.allergen),      # ... and names something the guest avoids
)
show("DANGEROUS (from the derived warnings)", dangerous_by_rule)

**Solution note.** 13 warnings: 5 milk, 1 fish, 7 gluten. The join finds the same violation as 3(b), and it
also names the allergen. Both ways are legitimate and both exist in real systems:

- the predicate is *computed on demand* (a view in a database, backward chaining in Prolog, a "computable"
  in KnowRob): always up to date, paid for at every query;
- the rule *materialises* its conclusions (forward chaining, a materialised view, what the OWL reasoner did
  when it filled the inferred class hierarchy): fast to query, but stale as soon as the world changes. Task 4
  comes back to this.

### 3(d) Rules with exceptions

The menu needs a label for each pizza. The chef's rule of thumb:

> A pizza without meat and fish is **vegetarian**. Unless it has no cheese either: then it is **vegan**.
> Otherwise, if it has meat, it is **meaty**.

A rule tree states exactly this. Inside `with query:` the first `add` is the default conclusion,
`refinement(condition)` is the *unless* (it replaces the default where the condition also holds), and
`alternative(condition)` is the *otherwise* (tried where the query's own condition fails).

**Predict** the label of each of the seven pizzas, then run the cell. Which pizza gets no label at all?

In [ ]:
@dataclass(eq=False, repr=False)
class Label:
    pizza: Pizza

    def __repr__(self):
        return f"{type(self).__name__}({self.pizza})"

class Vegetarian(Label): ...
class Vegan(Label): ...
class Meaty(Label): ...


def label_pizzas():
    label = deduced_variable(Label)
    query = entity(label).where(is_vegetarian(p))
    with query:
        add(label, inference(Vegetarian)(pizza=p))
        with refinement(not_(is_cheesy(p))):
            add(label, inference(Vegan)(pizza=p))
        with alternative(is_meaty(p)):
            add(label, inference(Meaty)(pizza=p))
    return query.tolist()


print(label_pizzas())

**Solution note.** `Vegetarian`: `margherita_1`, `mushroom_1`, `unclosed_1`. `Vegan`: `veggie_1`, `plain_1`.
`Meaty`: `american_hot_1`. **No label:** `napoletana_1`. It has fish, so it is not vegetarian, and it has no
meat, so the alternative does not fire either.

Two things to take away:

- *No label is not a negative label.* The rules are silent about the Napoletana; they do not say "not
  vegetarian". Whether silence means "no" is the closed-world question again.
- The refinement is an *exception*: the general conclusion (vegetarian) is withdrawn where a more specific
  one (vegan) applies. Classical logic and OWL cannot withdraw anything; they are monotonic. Rule trees of
  this shape are called ripple-down rules.

**Your turn (short).** `unclosed_1` was labelled vegetarian. Last week the OWL reasoner refused to say that.
Now the cook puts ham on it. Run the cell, then answer on the worksheet: which conclusion changed, and what
would an OWL reasoner have had to do?

In [ ]:
print("before:", [label for label in label_pizzas() if label.pizza is unclosed_1])

unclosed_1.toppings.append(HamTopping())
print("after: ", [label for label in label_pizzas() if label.pizza is unclosed_1])

unclosed_1.toppings.pop()   # the guest complained; the ham is removed again

**Solution note.** Before: `Vegetarian(unclosed_1)`. After: `Meaty(unclosed_1)`. New information *removed* a
conclusion: this is non-monotonic reasoning, the price of the closed-world assumption. The OWL reasoner never
has to retract, because under the open-world assumption it did not conclude "vegetarian" in the first place;
it answered "unknown" until the closure axiom was added. Note also that the label objects created "before"
still exist in Python. Nothing deletes a materialised conclusion automatically; keeping derived facts
consistent with a changing world is called truth maintenance.

---
## Task 4 · Live objects and recorded data

So far the knowledge base was the *live* state of the kitchen: Python objects that change while we look. A
restaurant also keeps *records*. At midnight the evening is written to a relational database, where it stays
as it was. The next cell does that with SQLite (run it; reading it is optional).

In [ ]:
def record_evening(orders: list[Order], pizzas: list[Pizza]) -> sqlite3.Connection:
    # Write tonight's pizzas and orders into a fresh relational database.
    database = sqlite3.connect(":memory:")
    database.executescript('''
        CREATE TABLE pizza      (id INTEGER PRIMARY KEY, name TEXT, kind TEXT);
        CREATE TABLE topping    (id INTEGER PRIMARY KEY, pizza_id INTEGER REFERENCES pizza(id),
                                 kind TEXT, spiciness TEXT);
        CREATE TABLE order_item (table_number INTEGER, guest TEXT, pizza_id INTEGER REFERENCES pizza(id));
        CREATE TABLE subclass_of (child TEXT, parent TEXT);
    ''')
    pizza_ids = {}
    for pizza in pizzas:
        cursor = database.execute("INSERT INTO pizza (name, kind) VALUES (?, ?)",
                                  (pizza.name, type(pizza).__name__))
        pizza_ids[pizza.name] = cursor.lastrowid
        for topping in pizza.toppings:
            database.execute("INSERT INTO topping (pizza_id, kind, spiciness) VALUES (?, ?, ?)",
                             (cursor.lastrowid, type(topping).__name__, topping.spiciness.name))
    for order in orders:
        for pizza in order.pizzas:
            database.execute("INSERT INTO order_item VALUES (?, ?, ?)",
                             (order.table, order.guest.name, pizza_ids[pizza.name]))
    topping_classes = [PizzaTopping]
    for topping_class in topping_classes:              # the class hierarchy, written down as rows
        for subclass in topping_class.__subclasses__():
            database.execute("INSERT INTO subclass_of VALUES (?, ?)",
                             (subclass.__name__, topping_class.__name__))
            topping_classes.append(subclass)
    database.commit()
    return database


def show_sql(title, database, statement):
    if statement is ...:
        print(f"{title}: TODO")
        return
    rows = database.execute(statement).fetchall()
    print(f"{title}: {len(rows)} row(s)")
    for row in rows:
        print("   ", row)


record = record_evening(orders, pizzas)
show_sql("table topping, first rows", record, "SELECT * FROM topping LIMIT 4")

### 4(a) The same question in SQL

*"Which pizzas have a hot topping?"* In EQL: `a(Pizza)(toppings=a(PizzaTopping)(spiciness=Spiciness.HOT))`.
In SQL the list `toppings` has become a second table, and the link between a pizza and its toppings has to
be spelled out as a join:

In [ ]:
show_sql("hot pizzas", record, '''
    SELECT DISTINCT pizza.name
    FROM pizza JOIN topping ON topping.pizza_id = pizza.id
    WHERE topping.spiciness = 'HOT'
''')

**Your turn.** Write `cheesy_sql`: the names of the pizzas with a cheese topping. Try the obvious condition
`topping.kind = 'CheeseTopping'` first and look at the result. Then explain it, and repair the query with
what you find in the table `subclass_of` (`SELECT * FROM subclass_of` shows it).

In [ ]:
obvious_sql = '''
    SELECT DISTINCT pizza.name
    FROM pizza JOIN topping ON topping.pizza_id = pizza.id
    WHERE topping.kind = 'CheeseTopping'
'''
show_sql("cheesy pizzas, obvious attempt", record, obvious_sql)

cheesy_sql = '''
    WITH RECURSIVE cheese(kind) AS (
        SELECT 'CheeseTopping'
        UNION
        SELECT subclass_of.child FROM subclass_of JOIN cheese ON subclass_of.parent = cheese.kind
    )
    SELECT DISTINCT pizza.name
    FROM pizza JOIN topping ON topping.pizza_id = pizza.id
    WHERE topping.kind IN (SELECT kind FROM cheese)
'''
show_sql("cheesy pizzas", record, cheesy_sql)

**Solution note.** The obvious query returns nothing. The rows say `MozzarellaTopping`; that mozzarella *is
a* cheese is knowledge about classes (the TBox), and a table of facts does not contain it. In Python the
class hierarchy is part of the program, so `a(CheeseTopping)()` matches a mozzarella without further ado.

To repair the SQL query, the hierarchy has to be stored as data (`subclass_of`) and the query has to compute
its transitive closure with a recursive common table expression. For this one-level hierarchy
`WHERE topping.kind IN (SELECT child FROM subclass_of WHERE parent = 'CheeseTopping')` is enough, but it
breaks as soon as someone adds `BuffaloMozzarellaTopping` below `MozzarellaTopping`. Recursion is what
Datalog and Prolog are built around, and what a `SELECT` of plain relational algebra cannot express.

### 4(b) The world moves on

It is busy. The cook mixes up two tickets and grates parmesan over `veggie_1`, the pizza for Ben at table 2.

The next cell asks three sources whether `veggie_1` has cheese: the query `cheesy`, which you ran at the very
beginning of Task 2; the record; and the milk warnings that the rule of 3(c) derived.

**Predict** the three outputs. Which of them would have protected Ben?

In [ ]:
veggie_1.toppings.append(ParmesanTopping())

show("1. cheesy pizzas, the query of 2(a) on the live objects", cheesy)
show_sql("2. pizzas with parmesan, the record", record, '''
    SELECT pizza.name FROM pizza JOIN topping ON topping.pizza_id = pizza.id
    WHERE topping.kind = 'ParmesanTopping'
''')
print("3. milk warnings derived in 3(c):", milk_warnings)

Run the rule again and compare with output 3:

In [ ]:
milk_warnings = inference(AllergenWarning)(pizza=cheesy, allergen=Allergen.MILK).tolist()
print("milk warnings, derived again:", milk_warnings)

**Solution note.** Output 1 now contains `veggie_1`; outputs 2 and 3 do not. Only the query over the live
objects would have protected Ben. With your predicate of 3(b), `show("DANGEROUS", dangerous)` now reports
two rows for table 2.

- A query over live objects is evaluated against the present state, every time it is evaluated. `cheesy`
  was written long before the parmesan; nothing had to be reloaded or synchronised.
- A record is a snapshot, and that is its purpose: the health inspector asks what was served on Thursday,
  not what the counter looks like now. To get the parmesan into it, somebody has to write it there.
- The warnings of 3(c) are in between: derived from the live objects, but at a fixed moment. They were
  already a small record, and a wrong one after the change, until the rule ran again. Deciding *when* to
  re-derive is the hard part of forward chaining; robots, whose world changes all the time, face it
  constantly.

### 4(c) What SQL cannot ask (here)

**Your turn.** Try to write the constraint of 3(b), *"the orders with a pizza that is not safe for the
guest"*, as SQL over `record`. You will get stuck. Note on the worksheet what exactly is missing: which
facts are not in any table, and which part of the query SQL has no word for?

In [ ]:
# Missing in the record: what each guest avoids (it was never written down),
# and which topping kind contains which allergen (that knowledge lived in the Python function allergens).
# With both added as tables, SQL can express the constraint:

record.executescript('''
    CREATE TABLE avoids   (guest TEXT, allergen TEXT);
    CREATE TABLE contains (topping_kind TEXT, allergen TEXT);
    INSERT INTO avoids   VALUES ('Ben', 'milk'), ('Chidi', 'fish');
    INSERT INTO contains VALUES ('CheeseTopping', 'milk'), ('FishTopping', 'fish');
''')

dangerous_sql = '''
    WITH RECURSIVE ancestor(kind, ancestor_kind) AS (
        SELECT child, child FROM subclass_of
        UNION
        SELECT ancestor.kind, subclass_of.parent
        FROM ancestor JOIN subclass_of ON subclass_of.child = ancestor.ancestor_kind
    )
    SELECT DISTINCT order_item.table_number, order_item.guest, pizza.name, avoids.allergen
    FROM order_item
    JOIN pizza    ON pizza.id = order_item.pizza_id
    JOIN topping  ON topping.pizza_id = pizza.id
    JOIN ancestor ON ancestor.kind = topping.kind
    JOIN contains ON contains.topping_kind = ancestor.ancestor_kind
    JOIN avoids   ON avoids.guest = order_item.guest AND avoids.allergen = contains.allergen
'''
show_sql("dangerous orders", record, dangerous_sql)

**Solution note.** SQL is not too weak for this; the *record* is too poor. Everything a SQL query uses must
be a row in a table. Our record had no table for what guests avoid, and the knowledge "cheese contains milk"
lived inside a Python function. Once both are tables, five joins and a recursive expression do the job
(result: table 2, Ben, `napoletana_1`, milk). The EQL version was one line, because it could follow object
references (`o.guest.avoids`) and call the function.

The trade is real in both directions. The database keeps millions of rows on disk, answers with indexes,
survives a power cut and serves many users at once; an EQL query over Python objects does none of that.
KRROOD's ORMatic bridges the two: it generates the tables from the dataclasses and translates a part of EQL
into SQL, so that one query can be asked of the live objects and of the record.

---
## Bonus · A query is an object

These cells need the optional package `casadi`; skip them if the import fails.

A query is a data structure, so a program can do more with it than evaluate it. It can **say** it:

In [ ]:
from krrood.entity_query_language.verbalization.pipeline import verbalize_expression

print(verbalize_expression(entity(p).where(length(p.toppings) >= 3)))
print(verbalize_expression(set_of(g, p).where(IsSafeFor(p, g))))

And it can be used the other way round. Until now a description *selected* among existing objects. A guest
who orders *"a pizza with mozzarella and a hot green pepper, any pepper strength"* describes a pizza that
does not exist yet. A generative backend **constructs** the objects that fit the description; `...` marks
what the guest left open.

In [ ]:
from krrood.entity_query_language.backends import EntityQueryLanguageGenerativeBackend

wish = a(Pizza)(
    name="for table 4",
    toppings=[a(MozzarellaTopping)(spiciness=Spiciness.MILD), a(HotGreenPepperTopping)(spiciness=...)],
)
for pizza in EntityQueryLanguageGenerativeBackend().evaluate(wish):
    print(pizza, [(topping, topping.spiciness.name) for topping in pizza.toppings])

**Solution note.** The same expression was a question in Task 2 (which objects fit?), a constraint in Task 3
(nothing may fit) and is a specification here (make something that fits). Three pizzas are constructed, one
per spiciness. This is the idea behind underspecified robot plans: *"pick up a cup"* describes the action,
and the system fills in which cup, which arm and which grasp.